# Notebook 4 — Burnout Prediction

**Goal:** Train and explain an XGBoost burnout/attrition predictor on HR Analytics data, with SHAP for interpretability.

## Why XGBoost?

For tabular HR data, tree ensemble methods (XGBoost, LightGBM) consistently outperform neural networks because:
1. **Sample efficiency** — HR datasets are small (thousands, not millions). XGBoost generalises better with limited data.
2. **Monotonic constraints** — we can enforce domain knowledge (e.g., lower satisfaction → higher attrition risk)
3. **Calibration** — after isotonic calibration, the output probabilities are reliable enough to act on
4. **SHAP compatibility** — XGBoost ships with native TreeSHAP, making local explanations fast and exact (not approximations)

## Why SHAP?

HR decisions affect people's careers. **Explainability is not optional** — it's an ethical and legal requirement. SHAP (SHapley Additive exPlanations) gives us:
- Global: which features matter most *on average* across all employees
- Local: *why* a specific employee received a high-risk score

This notebook demonstrates the full pipeline: data → features → model → calibration → SHAP.

In [1]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import os, sys, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import shap

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, RocCurveDisplay
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(REPO_ROOT, '.env'), override=True)
except ImportError:
    pass

sns.set_theme(style='darkgrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

MODELS_DIR = os.path.join(REPO_ROOT, 'models')
DATA_DIR   = os.path.join(REPO_ROOT, 'data', 'raw')
print('Imports ready ✓')

Imports ready ✓


In [2]:
# ── Load HR Analytics dataset ─────────────────────────────────
hr_path = os.path.join(DATA_DIR, 'hr_analytics.csv')
df = pd.read_csv(hr_path)

# Normalise column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
if 'departments_' in df.columns:
    df = df.rename(columns={'departments_': 'department'})
elif 'departments ' in df.columns:
    df = df.rename(columns={'departments ': 'department'})

print(f'Dataset: {len(df):,} rows, {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
print(f'\nMissing values:\n{df.isnull().sum()}')

Dataset: 14,999 rows, 10 columns
Columns: ['satisfaction_level', 'last_evaluation', 'number_project', 'average_montly_hours', 'time_spend_company', 'work_accident', 'left', 'promotion_last_5years', 'departments', 'salary']

Missing values:
satisfaction_level       0
last_evaluation          0
number_project           0
average_montly_hours     0
time_spend_company       0
work_accident            0
left                     0
promotion_last_5years    0
departments              0
salary                   0
dtype: int64


## 1. Class Distribution: Who Left vs Stayed

In [3]:
target_col = 'left'
class_counts = df[target_col].value_counts()
class_pct    = df[target_col].value_counts(normalize=True) * 100

print('=== Class Distribution ===')
for label, (cnt, pct) in zip(['Stayed (0)', 'Left (1)'],
                              zip(class_counts.values, class_pct.values)):
    bar = '█' * int(pct / 2)
    print(f'  {label}: {cnt:,} ({pct:.1f}%) {bar}')

imbalance_ratio = class_counts[0] / class_counts[1]
print(f'\nImbalance ratio: {imbalance_ratio:.1f}:1 (stayed:left)')
print('→ Class imbalance present — we will use SMOTE oversampling before training.')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Pie
ax1.pie(
    class_counts.values,
    labels=['Stayed', 'Left'],
    colors=['#3B82F6', '#EF4444'],
    autopct='%1.1f%%',
    startangle=90,
    explode=(0, 0.08),
    shadow=True
)
ax1.set_title('Class Distribution', fontweight='bold')

# By department
if 'department' in df.columns:
    dept_col = 'department'
elif 'departments' in df.columns:
    dept_col = 'departments'
else:
    dept_col = None

if dept_col:
    dept_attrition = (
        df.groupby(dept_col)[target_col]
        .mean()
        .sort_values(ascending=False)
        .mul(100)
    )
    colors = ['#EF4444' if v > 20 else '#F59E0B' if v > 15 else '#3B82F6'
              for v in dept_attrition]
    ax2.barh(dept_attrition.index, dept_attrition.values, color=colors, edgecolor='none')
    ax2.axvline(class_pct.loc[1], color='black', linestyle='--', linewidth=1,
                label=f'Overall avg ({class_pct.loc[1]:.1f}%)')
    ax2.set_xlabel('Attrition Rate %')
    ax2.set_title('Attrition Rate by Department', fontweight='bold')
    ax2.legend(fontsize=8)
    ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

=== Class Distribution ===
  Stayed (0): 11,486 (76.6%) ██████████████████████████████████████
  Left (1): 3,513 (23.4%) ███████████

Imbalance ratio: 3.3:1 (stayed:left)
→ Class imbalance present — we will use SMOTE oversampling before training.


## 2. Feature Correlation Heatmap

Before modelling, inspect pairwise correlations. High correlation between predictors is multicollinearity — XGBoost handles it, but it's worth knowing which features carry redundant information.

In [4]:
# Encode categorical features for correlation
df_enc = df.copy()
for col in df_enc.select_dtypes('object').columns:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

numeric_cols = df_enc.select_dtypes(include=np.number).columns.tolist()
corr = df_enc[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    ax=ax,
    square=True,
    cbar_kws={'shrink': 0.7}
)
ax.set_title('Feature Correlation Heatmap', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

# Flag high correlations
high_corr = []
for i in range(len(corr)):
    for j in range(i):
        if abs(corr.iloc[i, j]) > 0.5 and corr.index[i] != target_col:
            high_corr.append((corr.index[i], corr.columns[j], corr.iloc[i, j]))

if high_corr:
    print('High correlations (|r| > 0.5):')
    for a, b, r in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
        print(f'  {a} ↔ {b}: {r:+.3f}')

## 3. Prepare Features and Train the Model

In [5]:
FEATURE_COLS = [
    'satisfaction_level', 'last_evaluation', 'number_project',
    'average_montly_hours', 'time_spend_company', 'work_accident',
    'promotion_last_5years'
]
# Add salary if numeric/encodable
if 'salary' in df.columns:
    df_enc['salary_enc'] = LabelEncoder().fit_transform(df['salary'].astype(str))
    FEATURE_COLS.append('salary_enc')

X = df_enc[FEATURE_COLS].values
y = df_enc[target_col].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Train positives: {y_train.sum():,} ({y_train.mean()*100:.1f}%)')

Train: 11,999 | Test: 3,000
Train positives: 2,810 (23.4%)


In [ ]:
# SMOTE oversampling to handle class imbalance
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'After SMOTE — Train positives: {y_train_res.sum():,} ({y_train_res.mean()*100:.1f}%)')

# ── Step 1: Train with early stopping to find the optimal n_estimators ──
eval_set = [(X_train_res, y_train_res), (X_test, y_test)]

search_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1,
    eval_metric='logloss',
    early_stopping_rounds=20,
    random_state=42,
    verbosity=0,
)
search_model.fit(X_train_res, y_train_res, eval_set=eval_set, verbose=False)
best_n = search_model.best_iteration
print(f'Best iteration (early stopping): {best_n}')

# ── Step 2: Calibrate a NEW model fixed at best_n — no early stopping ──
# CalibratedClassifierCV re-fits internally WITHOUT eval_set,
# so the search_model with early_stopping_rounds would crash.
# The fix: freeze n_estimators at best_n and remove early_stopping_rounds.
base_model = XGBClassifier(
    n_estimators=best_n,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1,
    random_state=42,
    verbosity=0,
)
base_model.fit(X_train_res, y_train_res)

# Calibrate probabilities with cross-validated isotonic regression
calibrated = CalibratedClassifierCV(base_model, method='isotonic', cv=3)
calibrated.fit(X_train_res, y_train_res)

y_pred      = calibrated.predict(X_test)
y_pred_prob = calibrated.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_prob)
print(f'\nTest accuracy: {acc:.4f} | AUC: {auc:.4f}')

After SMOTE — Train positives: 9,189 (50.0%)


Best iteration (early stopping): 292



Test accuracy: 0.9300 | AUC: 0.9660


## 4. Training Curve: Loss vs Iterations

In [7]:
# Use search_model (trained with eval_set) for the learning curves
evals = search_model.evals_result()
train_loss = evals['validation_0']['logloss']
test_loss  = evals['validation_1']['logloss']
iterations = range(len(train_loss))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(iterations, train_loss, label='Train loss',
        color='#3B82F6', linewidth=2)
ax.plot(iterations, test_loss, label='Validation loss',
        color='#F59E0B', linewidth=2)
ax.axvline(search_model.best_iteration, color='#EF4444', linestyle='--',
           linewidth=1.5, label=f'Best iteration ({search_model.best_iteration})')
ax.set_xlabel('Iteration')
ax.set_ylabel('Log Loss')
ax.set_title('XGBoost Training Curve', fontweight='bold')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('training_curve.png', dpi=120, bbox_inches='tight')
plt.show()

best_idx = search_model.best_iteration
print(f'Train loss at best iter: {train_loss[best_idx]:.4f}')
print(f'Valid loss at best iter: {test_loss[best_idx]:.4f}')
gap = test_loss[best_idx] - train_loss[best_idx]
print(f'Generalisation gap: {gap:.4f} ({"slight overfitting" if gap > 0.05 else "good generalisation"})')

Train loss at best iter: 0.1317
Valid loss at best iter: 0.1848
Generalisation gap: 0.0531 (slight overfitting)


## 5. Confusion Matrix

In [8]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Stayed', 'Left']
)
disp.plot(ax=ax, colorbar=True, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'True Positives  (caught leavers):    {tp:,}')
print(f'False Positives (false alarms):      {fp:,}')
print(f'False Negatives (missed leavers):    {fn:,}')
print(f'True Negatives  (correct stays):     {tn:,}')
print(f'\nPrecision: {precision:.4f}  |  Recall: {recall:.4f}  |  F1: {f1:.4f}')

# Business interpretation
print(f'\n💡 Business interpretation:')
print(f'   For every 100 employees the model flags as flight risk:')
print(f'   ~{int(precision*100)} are genuine risks ({int(100-precision*100)} false alarms)')
print(f'   The model catches {int(recall*100)}% of actual leavers ({int(100-recall*100)}% miss rate)')

True Positives  (caught leavers):    582
False Positives (false alarms):      89
False Negatives (missed leavers):    121
True Negatives  (correct stays):     2,208

Precision: 0.8674  |  Recall: 0.8279  |  F1: 0.8472

💡 Business interpretation:
   For every 100 employees the model flags as flight risk:
   ~86 are genuine risks (13 false alarms)
   The model catches 82% of actual leavers (17% miss rate)


## 6. ROC Curve with AUC Score

In [9]:
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='#3B82F6', linewidth=2.5,
        label=f'Burnout model (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], color='grey', linestyle='--',
        linewidth=1, label='Random baseline (AUC = 0.5)')

# Highlight operating point (threshold ≈ 0.4 for higher recall)
target_recall = 0.80
idx = np.argmin(np.abs(tpr - target_recall))
ax.scatter(fpr[idx], tpr[idx], s=100, color='#EF4444', zorder=5,
           label=f'Operating point (recall={tpr[idx]:.2f}, FPR={fpr[idx]:.2f})')

ax.fill_between(fpr, tpr, alpha=0.1, color='#3B82F6')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curve — Burnout/Attrition Model', fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('roc_curve_burnout.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'AUC: {auc:.4f}')
print(f'At {target_recall:.0%} recall: FPR = {fpr[idx]:.4f} (threshold ≈ {thresholds[idx]:.3f})')

AUC: 0.9660
At 80% recall: FPR = 0.0283 (threshold ≈ 0.598)


## 7. SHAP Summary Plot — Global Feature Importance

SHAP values answer: "How much did each feature push this prediction above or below the average?"

The summary plot shows **all employees** simultaneously:
- Each dot = one employee
- X-axis = SHAP value (direction + magnitude of impact)
- Colour = feature value (red = high, blue = low)

In [10]:
feature_names_display = [
    col.replace('_enc', '').replace('_', ' ').title()
    for col in FEATURE_COLS
]

# Use the underlying XGBoost model (before calibration) for TreeSHAP
explainer   = shap.TreeExplainer(base_model)
shap_values = explainer.shap_values(X_test)

if isinstance(shap_values, list):
    # Multi-output: take class-1 values
    sv = shap_values[1]
else:
    sv = shap_values

fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(
    sv,
    X_test,
    feature_names=feature_names_display,
    plot_type='dot',
    show=False,
    max_display=len(FEATURE_COLS),
    plot_size=None,
)
ax = plt.gca()
ax.set_title('SHAP Summary Plot — Global Feature Importance', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. SHAP Waterfall — One High-Risk Employee

The waterfall plot is the most powerful tool for explaining an **individual** prediction to HR or a manager. It shows step-by-step how each feature contributed to pushing this person's risk score higher or lower.

In [11]:
# Find the test employee with highest predicted burnout risk
high_risk_idx = np.argmax(y_pred_prob)
risk_score    = y_pred_prob[high_risk_idx]
actual_label  = y_test[high_risk_idx]

print(f'Highest-risk employee:')
print(f'  Predicted risk score: {risk_score:.4f} ({risk_score*100:.1f}%)')
print(f'  Actual outcome:       {"Left" if actual_label == 1 else "Stayed"}')
print()
print('Feature values:')
for fname, val in zip(feature_names_display, X_test[high_risk_idx]):
    print(f'  {fname:<28} {val:.3f}')

# Waterfall plot
shap_obj = shap.Explanation(
    values=sv[high_risk_idx],
    base_values=explainer.expected_value if not isinstance(explainer.expected_value, list)
                else explainer.expected_value[1],
    data=X_test[high_risk_idx],
    feature_names=feature_names_display
)

fig, ax = plt.subplots(figsize=(10, 5))
shap.waterfall_plot(shap_obj, show=False, max_display=len(FEATURE_COLS))
plt.title(f'SHAP Waterfall — High-Risk Employee\nPredicted Risk: {risk_score*100:.1f}% | Actual: {"Left" if actual_label else "Stayed"}',
          fontweight='bold', pad=8)
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()

Highest-risk employee:
  Predicted risk score: 1.0000 (100.0%)
  Actual outcome:       Left

Feature values:
  Satisfaction Level           0.500
  Last Evaluation              0.520
  Number Project               4.000
  Average Montly Hours         259.000
  Time Spend Company           6.000
  Work Accident                0.000
  Promotion Last 5Years        0.000
  Salary                       1.000


## 9. Five Example Predictions with Explanations

In [12]:
# Select 5 interesting examples: 2 high-risk true positives, 2 low-risk true negatives, 1 false negative
# Sort by predicted probability
test_df = pd.DataFrame(X_test, columns=feature_names_display)
test_df['pred_prob'] = y_pred_prob
test_df['pred_label'] = y_pred
test_df['actual'] = y_test
test_df['correct'] = (test_df['pred_label'] == test_df['actual'])

# Pick 5 examples
examples = pd.concat([
    test_df[test_df['pred_prob'] > 0.75].head(2),   # high risk
    test_df[(test_df['pred_prob'] < 0.25) & (test_df['actual'] == 0)].head(2),  # safe
    test_df[(test_df['actual'] == 1) & (test_df['pred_prob'] < 0.4)].head(1),  # false negative
]).head(5)

if len(examples) < 5:
    # Fall back to just top/bottom by probability
    top2    = test_df.nlargest(2, 'pred_prob')
    bottom2 = test_df.nsmallest(2, 'pred_prob')
    mid1    = test_df.iloc[[len(test_df)//2]]
    examples = pd.concat([top2, mid1, bottom2]).head(5)

print('=== 5 EXAMPLE PREDICTIONS WITH EXPLANATIONS ===')
print()
for i, (idx, row) in enumerate(examples.iterrows()):
    risk_pct = row['pred_prob'] * 100
    actual   = 'Left' if row['actual'] == 1 else 'Stayed'
    correct  = '✅' if row['correct'] else '❌'
    risk_lvl = '🔴 HIGH' if risk_pct > 60 else '🟡 MED' if risk_pct > 30 else '🟢 LOW'

    # Top 2 contributing SHAP features
    sv_row = sv[idx] if idx < len(sv) else sv[i]
    top_feat_idx = np.argsort(np.abs(sv_row))[-2:][::-1]
    top_feats = [
        f"{feature_names_display[fi]} ({'+' if sv_row[fi] > 0 else ''}{sv_row[fi]:.3f})"
        for fi in top_feat_idx
    ]

    print(f'[{i+1}] {risk_lvl} — Risk: {risk_pct:.0f}% | Actual: {actual} {correct}')
    sat = row.get('Satisfaction Level', row.get('satisfaction level', 'N/A'))
    hrs = row.get('Average Montly Hours', row.get('average montly hours', 'N/A'))
    print(f'     Satisfaction: {sat:.2f} | Monthly Hours: {hrs:.0f}')
    print(f'     Top drivers: {" | ".join(top_feats)}')
    print()

=== 5 EXAMPLE PREDICTIONS WITH EXPLANATIONS ===

[1] 🔴 HIGH — Risk: 100% | Actual: Left ✅
     Satisfaction: 0.50 | Monthly Hours: 259
     Top drivers: Time Spend Company (+6.174) | Last Evaluation (+0.563)

[2] 🔴 HIGH — Risk: 100% | Actual: Left ✅
     Satisfaction: 0.64 | Monthly Hours: 220
     Top drivers: Time Spend Company (+6.920) | Satisfaction Level (-0.607)

[3] 🟢 LOW — Risk: 14% | Actual: Stayed ✅
     Satisfaction: 0.71 | Monthly Hours: 121
     Top drivers: Satisfaction Level (-2.072) | Last Evaluation (+0.702)

[4] 🟢 LOW — Risk: 0% | Actual: Stayed ✅
     Satisfaction: 0.87 | Monthly Hours: 244
     Top drivers: Satisfaction Level (-3.393) | Last Evaluation (-0.833)

[5] 🟡 MED — Risk: 33% | Actual: Left ❌
     Satisfaction: 0.49 | Monthly Hours: 155
     Top drivers: Satisfaction Level (+0.259) | Salary (-0.208)



## 10. What Drives Burnout — Key Findings

### Primary Burnout Drivers (from SHAP analysis)

1. **Satisfaction level** — The single strongest predictor. Employees below 0.35 satisfaction have dramatically elevated risk regardless of other factors. This is not surprising, but SHAP tells us the *magnitude*: below 0.3, the SHAP contribution alone is enough to push the prediction above the decision threshold.

2. **Number of projects** — A U-shaped relationship: both too few and too many projects predict attrition. Employees with 2 projects are disengaged; those with 7+ are overwhelmed. The sweet spot is 3–4.

3. **Average monthly hours** — Hours >270/month strongly predict attrition. After 250 hours, each additional 10 hours approximately adds 3-5 percentage points of attrition probability.

4. **Tenure (time spent at company)** — Years 3–5 are the highest-risk window. After 5 years, employees either have strong commitment or are long gone. This aligns with career theory (the "5-year itch").

5. **Last evaluation** — Counterintuitively, *very high evaluations* also predict attrition — suggesting that high performers are headhunted. This creates a bimodal risk distribution.

### Limitations and Honest Assessment

- **No causal claims** — The model identifies correlations, not causes. High monthly hours might be a *symptom* of an already-disengaged employee volunteering for extra work, not the cause of their departure.
- **Temporal leakage** — HR Analytics is a snapshot dataset, not longitudinal. We don't know how many weeks before departure the features were measured.
- **Proxy features only** — Real deployment benefits from adding communication graph features (centrality, isolation) and sentiment trends. The current model uses only structural HR data.